Загрузка датасета

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

dataset_path = Path("/content/drive/MyDrive/diplom/dataset")

print([p.name for p in dataset_path.iterdir()])

Mounted at /content/drive
['video1', 'video2', 'video3', 'video4', 'video5', 'video6', 'video7', 'video8', 'video9', 'video10', 'video11', 'video12', 'video13', 'video14', 'video15']


In [ ]:
video_folders = sorted([p for p in dataset_path.iterdir() if p.is_dir()])

for folder in video_folders:
    print(f"\n📁 {folder.name}")
    print([p.name for p in folder.iterdir()])


📁 video1
['ru.srt', 'en.srt', 'video.mp4']

📁 video10
['video.mp4', 'ru.srt', 'en.srt']

📁 video11
['ru.srt', 'en.srt', 'video.mp4']

📁 video12
['ru.srt', 'en.srt', 'video.mp4']

📁 video13
['ru.srt', 'video.mp4', 'en.srt']

📁 video14
['ru.srt', 'en.srt', 'video.mp4']

📁 video15
['ru.srt', 'video.mp4', 'en.srt']

📁 video2
['ru.srt', 'en.srt', 'video.mp4']

📁 video3
['ru.srt', 'en.srt', 'video.mp4']

📁 video4
['ru.srt', 'en.srt', 'video.mp4']

📁 video5
['ru.srt', 'en.srt', 'video.mp4']

📁 video6
['ru.srt', 'video.mp4', 'en.srt']

📁 video7
['ru.srt', 'en.srt', 'video.mp4']

📁 video8
['ru.srt', 'en.srt', 'video.mp4']

📁 video9
['ru.srt', 'en.srt', 'video.mp4']


Извлечение аудио из всех видео

In [ ]:
import subprocess

def extract_audio(video_path, output_path):
    cmd = [
        "ffmpeg",
        "-y",
        "-i", str(video_path),
        "-ac", "1",
        "-ar", "16000",
        str(output_path)
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

Подготовка списка файлов

In [ ]:
data = []

for folder in video_folders:
    video_path = folder / "video.mp4"
    audio_path = folder / "audio.wav"

    extract_audio(video_path, audio_path)

    data.append({
        "folder": folder.name,
        "audio": audio_path,
        "ref": folder / "en.srt"
    })

print("Готово:", len(data), "видео")

Готово: 15 видео


Установка метрики WER

In [ ]:
!pip install jiwer
from jiwer import wer


Функция чтения srt

In [ ]:
import re

def read_srt_text(srt_path):
    lines = []
    with open(srt_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.isdigit():
                continue
            if "-->" in line:
                continue
            lines.append(line)
    return " ".join(lines)

def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
import time
import soundfile as sf

Wisper

In [ ]:
!pip install openai-whisper --quiet
!apt-get install ffmpeg -y -qq

import whisper
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Устройство:", device)

model = whisper.load_model("small", device=device)

Устройство: cpu


In [ ]:
import pandas as pd
import time
import soundfile as sf
from jiwer import wer

results_whisper = []

for item in data:
    print(f"\nОбрабатывается: {item['folder']}")

    ref_text = read_srt_text(item["ref"])
    ref_text = normalize_text(ref_text)

    audio_array, sr = sf.read(item["audio"])
    if audio_array.ndim == 2:
        audio_array = audio_array.mean(axis=1)
    audio_duration = len(audio_array) / sr

    start_time = time.time()
    result = model.transcribe(
        str(item["audio"]),
        language="en",
        task="transcribe",
        fp16=True if device == "cuda" else False
    )
    elapsed_time = time.time() - start_time

    hyp_text = normalize_text(result["text"])

    # метрика
    score = wer(ref_text, hyp_text)

    # коэффициент скорости
    speed_ratio = audio_duration / elapsed_time if elapsed_time > 0 else None

    results_whisper.append({
        "video": item["folder"],
        "audio_duration_sec": round(audio_duration, 2),
        "reference_len": len(ref_text.split()),
        "hypothesis_len": len(hyp_text.split()),
        "whisper_time_sec": round(elapsed_time, 2),
        "whisper_speed_ratio": round(speed_ratio, 2) if speed_ratio else None,
        "whisper_wer": score
    })

    print("WER:", score)
    print("Время распознавания:", round(elapsed_time, 2), "сек")
    print("Длительность аудио:", round(audio_duration, 2), "сек")
    print("Коэффициент скорости:", round(speed_ratio, 2) if speed_ratio else None)

df_whisper = pd.DataFrame(results_whisper)
df_whisper


Обрабатывается: video1
WER: 0.030108588351431393
Время распознавания: 510.11 сек
Длительность аудио: 868.91 сек
Коэффициент скорости: 1.7

Обрабатывается: video10
WER: 0.20710059171597633
Время распознавания: 66.75 сек
Длительность аудио: 143.31 сек
Коэффициент скорости: 2.15

Обрабатывается: video11
WER: 0.15142857142857144
Время распознавания: 193.02 сек
Длительность аудио: 294.15 сек
Коэффициент скорости: 1.52

Обрабатывается: video12
WER: 0.15555555555555556
Время распознавания: 22.71 сек
Длительность аудио: 30.09 сек
Коэффициент скорости: 1.33

Обрабатывается: video13
WER: 0.22277227722772278
Время распознавания: 66.2 сек
Длительность аудио: 159.34 сек
Коэффициент скорости: 2.41

Обрабатывается: video14
WER: 0.02040816326530612
Время распознавания: 27.65 сек
Длительность аудио: 30.09 сек
Коэффициент скорости: 1.09

Обрабатывается: video15
WER: 0.17525773195876287
Время распознавания: 50.22 сек
Длительность аудио: 129.03 сек
Коэффициент скорости: 2.57

Обрабатывается: video2
WER: 

,video,audio_duration_sec,reference_len,hypothesis_len,whisper_time_sec,whisper_speed_ratio,whisper_wer
0,video1,868.91,2026,1983,510.11,1.70,0.030109
1,video10,143.31,169,151,66.75,2.15,0.207101
2,video11,294.15,700,732,193.02,1.52,0.151429
3,video12,30.09,45,51,22.71,1.33,0.155556
4,video13,159.34,202,179,66.20,2.41,0.222772
5,video14,30.09,98,100,27.65,1.09,0.020408
6,video15,129.03,97,99,50.22,2.57,0.175258
7,video2,355.54,833,824,275.84,1.29,0.073229
8,video3,1166.66,2749,2771,784.94,1.49,0.097490
9,video4,864.08,2141,2159,512.98,1.68,0.056983


In [ ]:
print("Средний WER Whisper:", df_whisper["whisper_wer"].mean())
print("Среднее время Whisper:", df_whisper["whisper_time_sec"].mean())
print("Средний коэффициент скорости Whisper:", df_whisper["whisper_speed_ratio"].mean())

Средний WER Whisper: 0.10584891144063537
Среднее время Whisper: 360.31066666666663
Средний коэффициент скорости Whisper: 1.7226666666666668


Wav2Vec2

In [ ]:
!pip install transformers torchaudio soundfile --quiet

import torch
import numpy as np
import soundfile as sf
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Устройство:", device)

processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
wav2vec_model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h").to(device)
wav2vec_model.eval()

Устройство: cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-960h
Key                        | Status  | 
---------------------------+---------+-
wav2vec2.masked_spec_embed | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Wav2Vec2ForCTC(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2GroupNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
          (activation): GELUActivation()
          (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
        )
        (1-4): 4 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projection): Linear(in_features=512, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder)

In [ ]:
def transcribe_wav2vec2(wav_path, sample_rate=16000, chunk_length_s=20.0):
    speech, sr = sf.read(wav_path)
    speech = speech.astype("float32")

    if speech.ndim == 2:
        speech = np.mean(speech, axis=1)

    if sr != sample_rate:
        print(f"Предупреждение: sample rate = {sr}, ожидалось {sample_rate}")

    total_samples = speech.shape[0]
    chunk_size = int(chunk_length_s * sample_rate)

    texts = []

    for start in range(0, total_samples, chunk_size):
        end = min(start + chunk_size, total_samples)
        chunk = speech[start:end]

        if len(chunk) < int(0.5 * sample_rate):
            continue

        inputs = processor(
            chunk,
            sampling_rate=sample_rate,
            return_tensors="pt",
            padding=True,
        )

        with torch.no_grad():
            logits = wav2vec_model(inputs.input_values.to(device)).logits

        predicted_ids = torch.argmax(logits, dim=-1)
        text_chunk = processor.batch_decode(predicted_ids)[0]

        texts.append(text_chunk.lower().strip())

    return " ".join(texts)

In [ ]:
import pandas as pd
import time
import soundfile as sf
from jiwer import wer

results_wav2vec2 = []

for item in data:
    print(f"\nОбрабатывается: {item['folder']}")

    ref_text = read_srt_text(item["ref"])
    ref_text = normalize_text(ref_text)

    audio_array, sr = sf.read(item["audio"])
    if audio_array.ndim == 2:
        audio_array = audio_array.mean(axis=1)
    audio_duration = len(audio_array) / sr

    start_time = time.time()
    hyp_text = transcribe_wav2vec2(str(item["audio"]))
    elapsed_time = time.time() - start_time

    hyp_text = normalize_text(hyp_text)

    score = wer(ref_text, hyp_text)

    speed_ratio = audio_duration / elapsed_time if elapsed_time > 0 else None

    results_wav2vec2.append({
        "video": item["folder"],
        "audio_duration_sec": round(audio_duration, 2),
        "reference_len": len(ref_text.split()),
        "hypothesis_len": len(hyp_text.split()),
        "wav2vec2_time_sec": round(elapsed_time, 2),
        "wav2vec2_speed_ratio": round(speed_ratio, 2) if speed_ratio else None,
        "wav2vec2_wer": score
    })

    print("WER:", score)
    print("Время распознавания:", round(elapsed_time, 2), "сек")
    print("Длительность аудио:", round(audio_duration, 2), "сек")
    print("Коэффициент скорости:", round(speed_ratio, 2) if speed_ratio else None)

df_wav2vec2 = pd.DataFrame(results_wav2vec2)
df_wav2vec2


Обрабатывается: video1
WER: 0.15992102665350444
Время распознавания: 309.19 сек
Длительность аудио: 868.91 сек
Коэффициент скорости: 2.81

Обрабатывается: video10
WER: 0.6331360946745562
Время распознавания: 41.41 сек
Длительность аудио: 143.31 сек
Коэффициент скорости: 3.46

Обрабатывается: video11
WER: 0.23285714285714285
Время распознавания: 84.27 сек
Длительность аудио: 294.15 сек
Коэффициент скорости: 3.49

Обрабатывается: video12
WER: 0.5333333333333333
Время распознавания: 8.42 сек
Длительность аудио: 30.09 сек
Коэффициент скорости: 3.58

Обрабатывается: video13
WER: 0.6386138613861386
Время распознавания: 46.12 сек
Длительность аудио: 159.34 сек
Коэффициент скорости: 3.45

Обрабатывается: video14
WER: 0.3673469387755102
Время распознавания: 9.35 сек
Длительность аудио: 30.09 сек
Коэффициент скорости: 3.22

Обрабатывается: video15
WER: 0.5257731958762887
Время распознавания: 37.86 сек
Длительность аудио: 129.03 сек
Коэффициент скорости: 3.41

Обрабатывается: video2
WER: 0.21848

,video,audio_duration_sec,reference_len,hypothesis_len,wav2vec2_time_sec,wav2vec2_speed_ratio,wav2vec2_wer
0,video1,868.91,2026,2045,309.19,2.81,0.159921
1,video10,143.31,169,133,41.41,3.46,0.633136
2,video11,294.15,700,739,84.27,3.49,0.232857
3,video12,30.09,45,47,8.42,3.58,0.533333
4,video13,159.34,202,179,46.12,3.45,0.638614
5,video14,30.09,98,88,9.35,3.22,0.367347
6,video15,129.03,97,80,37.86,3.41,0.525773
7,video2,355.54,833,819,109.28,3.25,0.218487
8,video3,1166.66,2749,2770,349.50,3.34,0.225173
9,video4,864.08,2141,2178,259.28,3.33,0.148529


In [ ]:
print("Средний WER Wav2Vec2:", df_wav2vec2["wav2vec2_wer"].mean())
print("Среднее время Wav2Vec2:", df_wav2vec2["wav2vec2_time_sec"].mean())
print("Средний коэффициент скорости Wav2Vec2:", df_wav2vec2["wav2vec2_speed_ratio"].mean())

Средний WER Wav2Vec2: 0.35939047396107254
Среднее время Wav2Vec2: 170.62066666666666
Средний коэффициент скорости Wav2Vec2: 3.336


In [ ]:
df_compare = df_whisper.merge(
    df_wav2vec2[["video", "wav2vec2_time_sec", "wav2vec2_speed_ratio", "wav2vec2_wer"]],
    on="video",
    how="inner"
)

df_compare

print("Средний WER Whisper:", df_compare["whisper_wer"].mean())
print("Средний WER Wav2Vec2:", df_compare["wav2vec2_wer"].mean())

print("Среднее время Whisper:", df_compare["whisper_time_sec"].mean())
print("Среднее время Wav2Vec2:", df_compare["wav2vec2_time_sec"].mean())

print("Средний коэффициент скорости Whisper:", df_compare["whisper_speed_ratio"].mean())
print("Средний коэффициент скорости Wav2Vec2:", df_compare["wav2vec2_speed_ratio"].mean())

df_compare
output_path = "/content/drive/MyDrive/diplom/asr_compare.csv"
df_compare.to_csv(output_path, index=False)
print("Сохранено:", output_path)
test_data = [data[0]]

Средний WER Whisper: 0.10584891144063537
Средний WER Wav2Vec2: 0.35939047396107254
Среднее время Whisper: 360.31066666666663
Среднее время Wav2Vec2: 170.62066666666666
Средний коэффициент скорости Whisper: 1.7226666666666668
Средний коэффициент скорости Wav2Vec2: 3.336
Сохранено: /content/drive/MyDrive/diplom/asr_compare.csv
